# Step 3: A2A Supply Chain Planner (Coordinator + Specialists)

In Step 1, we used A2A discovery + specialist selection (A2A Olympics).
In Step 2, we used A2A coordination (PEMDAS AST reducer routing sub-ops to specialists).

## Overview
### Goal: build a Coordinator agent that:
<ol>
<li><b>plans</b> (decides what work needs doing)</li>
<li><b>selects</b> specialists (based on skills + constraints)</li>
<li><b>executes</b> (iterates search under a compute budget)</li>
<li><b>validates</b> every step with contracts + scoring</li>
<li><b>produces</b> an auditable trace a human can agree with</li>
</ol>

The demo is supply-chain flavored because:<ul>
<li>Most people understand “raw materials → assembly → shipping → tariffs”</li>
<li>It’s timely</li>
<li>There’s a clear objective function (min cost + satisfy constraints)</li>
<li>The “why” is inspectable: every action has a measurable effect</li>
</ul>

### What We’re Demonstrating
What “Coordinator Intelligence” Means Here (No Handwaving)

We are not trying to prove the coordinator is “smart” by vibes.

We prove it by showing:

✅ It produces a plan (search strategy + candidate generation)</br>
✅ It makes decisions (tradeoffs + constraint handling)</br>
✅ It adapts to budget (more budget ⇒ better solutions, better confidence)</br>
✅ It creates a full trace (every hop has inputs/outputs/latency)</br>
✅ It never silently cheats (contracts + schemas on the wire)</br>

Success is not “best solution in the world.”</br>
Success is: a valid solution + a readable decision trail.

### Agents and Responsibilities
#### A2A Agents (Roles)
<b>Coordinator</b>
<ul><li>Inputs: scenario + constraints + budget</li>
<li>Outputs: best plan found + full trace</li>
<li>Chooses strategy (greedy/beam/random_restart/anneal) based on policy/adaptive settings</li>
</ul>
<b>Specialist Agents (skills)</b>
<ul><li>tariff.calc → compute duties given origin/destination/product class</li>
<li>route.enumerate → generate feasible next steps from a partial plan</li>
<li>plan.score → compute total landed cost + penalties</li>
<li>plan.validate → enforce hard constraints + explain violations</li>
<li>plan.repair (optional later) → attempt minimal edits to fix invalid plans
</ul>
In v1, these can be deterministic Python logic behind A2A endpoints.</br>
No LLM required.

### Message Format (Envelope + Typed Payload)
<b>Wire Contract + Payload Contract</b> (Same Pattern as PEMDAS)</br></br>
Each hop sends:</br>
<b>A2A Envelope</b>
<ul><li>source, dest, timestamp, trace_id, request_id</li>
<li>message_type</li>
<li>payload</li>
</ul></br>
<b>Typed Payload</b> - validated by JSON Schema per message_type</br>
This keeps the whole system honest:
<ul><li>specialists can be swapped without breaking protocol</li>
<li>coordinator can debug failures precisely</li>
<li>logs are structured and replayable</li>
</ul>

### Strategy Control: Selection Method + Strategy
<b>Strategy Control Vocabulary</b></br>
We use two terms:<br>

1) selection_method
<ul><li>fixed → force a specific strategy (testing)</li>
<li>policy → rules-based selection (configured)</li>
<li>adaptive → coordinator changes strategy mid-run (later)</li></ul>

2) strategy
<ul><li>greedy → best immediate improvement</li>
<li>beam → keep top-k candidates per depth</li>
<li>random_restart → escape local minima with multiple seeds</li>
<li>anneal → accept worse steps early; tighten later (optional)</li>
</ul>

<b>What's stored</b>:
<ul><li>strategy_requested (what user asked for; can be null)</li>
<li>strategy_selected (what we actually ran)</li>
<li>selection_reason (why)</li></ul></br>

**Example**

```json
{
  "selection_method": "adaptive",
  "strategy_requested": None,
  "strategy_selected": "beam",
  "selection_reason": "budget=200; need diversity + constraint satisfaction"
}
```

### Compute Budget and “Learning Curve”
<b>Budget Drives Behavior</b></br>
We expose compute_budget as a first-class parameter.

Interpretation:
<ul><li>small budgets: shallow search (greedy)</li>
<li>medium budgets: maintain diversity (beam)</li>
<li>large budgets: explore/escape minima (random_restart / anneal)</li></ul>
</br>
We do NOT hardcode:
“if budget < 20 use greedy”

Instead we implement a policy module:
<ul><li>look-up-table now (simple, explainable)</li>
<li>adaptive switching later (based on progress indicators)</li></ul>
</br>
<b>Key artifact for the post</b>:</br>
Show the same scenario solved with budgets 20 vs 200 vs 2000
and demonstrate solution quality improving + confidence increasing

### Execution Loop
#### <b>Coordinator Execution Loop (v1)</b></br>
<b>Inputs</b>:<ul>
<li>scenario data (suppliers, costs, lead times, tariffs)</li>
<li>constraints (must-haves)</li>
<li>objective weights (cost vs speed vs risk)</li>
<li>compute budget</li></ul>
</br>
<b>Algorithm</b>:<ol>
<li>initialize candidate set (seed plans)</li>
<li>repeat until budget exhausted:
<ul><li>expand candidates (route.enumerate)</li>
<li>score candidates (plan.score)</li>
<li>validate candidates (plan.validate)</li>
<li>keep best subset (strategy controls this)</li></ul></li>
<li>return best valid plan + explanation trace</li>
</ol></br>

<b>Outputs</b>:<ul>
<li>best plan found</li>
<li>total cost breakdown</li>
<li>validation status</li>
<li>full trace (calls + latency + deltas per step)</li>
</ul>

### Trace Format
Trace = Debuggable Coordination

Every iteration stores a trace event like:

```json
{
  "step": 12,
  "strategy_effective": "beam",
  "candidate_id": "cand-04",
  "action": "expand",
  "call": {
    "agent": "ROUTE_ENUM",
    "message_type": "route.enumerate:v1",
    "request_id": "...",
    "latency_ms": 18.2
  },
  "before": {"partial_plan": "..."},
  "after": {"new_candidates": 5}
}
```
</br>
This is the “aha”:<ul>
<li>coordination is not a black box</li>
<li>the coordinator is not “thinking,” it’s operating</li>
</ul>

### Milestones (Incremental Build Plan)
#### Build Plan (Incremental)
**Milestone 0 — Contracts**: define schemas for
<ul><li>scenario input</li>
<li>plan candidate</li>
<li>scoring output</li>
<li>validation result</li>
<li>coordinator request/response</li></ul>

**Milestone 1 — Specialists**: implement deterministic specialists as A2A services
<ul><li>validate</li>
<li>score</li>
<li>enumerate</li></ul>

**Milestone 2 — Coordinator v1**
<ul><li>fixed strategy (greedy)</li>
<li>returns a valid plan + trace</li></ul>

**Milestone 3 — Coordinator v1.1**
<ul><li>selection_method = fixed | policy</li>
<li>run same scenario under 20 / 200 / 2000 budget</li>
</ul>

**Milestone 4 — Adaptive**
<ul><li>switch strategy mid-run if progress stalls</li>
<li>store strategy_selected_initial/final</li></ul>

**Milestone 5 — Post assets**
<ul><li>banner diagram</li>
<li>trace snippet</li>
<li>comparison chart: budget vs best score</li>
</ul>

### Risk Management (aka “Reality”)
**Known Risk: Theory vs Practice**

Even with clean A2A plumbing, failures will happen:
<ul><li>schema drift (payload mismatch)</li>
<li>registry discovery issues (wrong agent, wrong endpoint)</li>
<li>search getting stuck in bad local minima</li>
<li>trace bloat / unreadable output</li></ul>
</br>
We mitigate by:
<ul><li>strict validation at every hop</li>
<li>small scenario first (3 suppliers, 2 routes)</li>
<li>consistent trace format</li>
<li>locked “fixed strategy” mode for debugging</li></ul>

As Yogi would say:

In theory, this is easy.
In practice, that’s why we have logs.

## Execution
### Milestone 0: Contract Map
**Contract Map (What each schema is for)**

We split contracts into two layers:

A) General A2A wire contracts (shared across projects)
<ul><li>a2a_request_v1.json</li>
<li>a2a_response_v1.json</li>
</ul>
These define the envelope (routing + traceability).

B) Project-specific payload contracts (this demo only)
<ul><li>supply_problem_v1.json</li>
<li>supply_plan_v1.json</li>
<li>supply_price_v1.json</li>
<li>supply_price_breakdown_v1.json</li>
<li>(optional but recommended) supply_trace_event_v1.json, supply_strategy_config_v1.json</li></ul>
</br>
These define the meaningful data that agents compute and pass around.

Envelope = “how messages travel”
Payload = “what the messages mean”

#### A2A Wire Contracts (Protocol Layer)

<b><i>a2a_request_v1.json</i></b>

**Purpose**: Standard wrapper for sending a typed message to an agent.

Used by: coordinator → specialists, user → coordinator

Key fields:
<ul><li>request_id (UUID): unique per request</li>
<li>trace_id (UUID): shared across the whole run</li>
<li>timestamp: when the request was created</li>
<li>source / dest: endpoints (who is calling who)</li>
<li>message_type: the “payload type”</li>
<li>payload: object validated by the schema implied by message_type</li></ul>

Mental model: “an addressed envelope containing typed data.”
</br></br>
<b><i>a2a_response_v1.json</i></b>

<b>Purpose</b>: Standard wrapper for returning results (or errors).

Used by: specialists → coordinator, coordinator → user</br>
Key fields:
<ul><li>request_id: matches the request</li>
<li>trace_id: same trace</li>
<li>timestamp: response time</li>
<li>ok: boolean
<ul><li>if ok=True: payload exists and is validated by message_type</li>
<li>if ok=False: error exists (structured)</li></ul>
</ul>
Mental model: “receipt + contents + error handling.”

#### Supply Chain Payload Contracts (Demo Layer)

These define what the supply-chain demo means in data.

<b><i>supply_problem_v1.json</i></b></br></br>
<b>Purpose</b>: The entire scenario definition and constraints.</br>
<b>Produced by</b>: user / notebook</br>
<b>Consumed by</b>: coordinator (and sometimes route.enumerate)</br>

**Typical contents**:<ul>
<li>suppliers: costs, countries, capacities</li>
<li>manufacturing options: allowed assembly countries, costs</li>
<li>shipping options: lanes + time/cost</li>
<li>tariff policy: duties by origin/destination/category (simplified table)</li>
<li>constraints:<ul>
<li>max lead time</li>
<li>min quality score</li>
<li>banned countries</li>
<li>min capacity</li></ul></li>
<li>objective weights:
<ul><li>landed_cost weight</li>
<li>speed weight</li>
<li>risk weight</li></ul></li>
</ul>
Why it exists: Without this contract, everyone “assumes” the scenario. That’s how demos lie.
</br></br>

<b><i>supply_plan_v1.json</i></b></br></br>
<b>Purpose</b>: A candidate plan (one possible supply chain).</br>
<b>Produced by</b>: coordinator and route.enumerate</br>
<b>Consumed by</b>: plan.score, plan.validate, coordinator (selection)</br>

**Contains**:<ul>
<li>selected supplier(s)</li>
<li>build/assembly location</li>
<li>shipping lane(s)</li>
<li>intermediate steps (optional)</li>
<li>derived properties (optional): total units, lead time, etc.</li>
</ul></br>
Key idea: This is the “state” the coordinator searches over.
</br></br>

<b><i>supply_price_breakdown_v1.json</i></b></br></br>
<b>Purpose</b>: A line-item cost explanation.</br>
<b>Produced by</b>: plan.score</br>
<b>Consumed by</b>: coordinator + user output</br>

**Typical fields**:<ul>
<li>material_cost</li>
<li>manufacturing_cost</li>
<li>shipping_cost</li>
<li>tariff_cost</li>
<li>handling_cost</li>
<li>total_landed_cost</li></ul></br>
plus “notes” for interpretability</br>
Why it matters: it turns “trust me” into “show your work.”</br></br>

<b><i>supply_price_v1.json</i></b></br></br>
<b>Purpose</b>: Score object used for ranking plans.</br>
<b>Produced by</b>: plan.score</br>
<b>Consumed by</b>: coordinator (strategy selection + pruning)</br>

**Usually contains**:<ul>
<li>total_score (lower is better)</li>
<li>is_valid or validity_status</li>
<li>breakdown: supply_price_breakdown_v1</li>
<li>penalties: list (soft constraint violations)</li></ul>
</br>
Tip: Keep breakdown separate because humans want it. Strategies want the scalar score.

#### Optional Contracts (High ROI for Debuggability)

<b><i>supply_trace_event_v1.json</i></b></br></br>
<b>Purpose</b>: Normalize trace events across all strategies.</br>
<b>Produced by</b>: coordinator (every step)</br>
<b>Consumed by</b>: final report renderer</br></br>

**Fields**:<ul>
<li>step index</li>
<li>strategy effective (greedy/beam/…)</li>
<li>candidate id</li>
<li>action type (expand/score/validate/select)</li>
<li>call metadata (agent, message_type, latency_ms)</li>
<li>before/after summaries (don’t bloat)</li></ul>
</br>
Why you want it: The trace becomes “product-quality,” not random dict soup.</br>

<b><i>supply_strategy_config_v1.json</b></i></br></br>
<b>Purpose</b> : Makes the coordinator’s strategy selection explicit.</br>
<b>Produced by</b>: user OR coordinator policy</br>
<b>Consumed by</b>: coordinator</br>

**Fields**:<ul>
<li>selection_method: fixed | policy | adaptive</li>
<li>strategy_requested: nullable</li>
<li>strategy_selected</li>
<li>selection_reason</li>
<li>compute_budget</li>
<li>strategy params (beam width, restarts, etc.)</li></ul>
</br>
Why you want it: Great for your post. You can show “budget changes behavior” cleanly.

### Message Types and Who Uses Them

**Coordinator receives**:
<ul><li>supply.solve:v1 → payload: supply_problem_v1</li></ul></br>

**Coordinator calls specialists**:
<ul><li>route.enumerate:v1 → returns more supply_plan_v1</li>
<li>plan.validate:v1 → returns validation result (bool + violations)</li>
<li>plan.score:v1 → returns supply_price_v1</li></ul></br>

**Coordinator returns**:
<ul><li>supply.solve.result:v1 → best plan + score + trace</li></ul></br>
</br>
The coordinator is basically a search engine over supply_plan_v1 states.

### What Makes This “Provably Not Fake”

We aren’t claiming the “perfect plan.”

We’re showing:
<ul><li>every plan is validated</li>
<li>every score is decomposed (breakdown)</li>
<li>every step is logged + timed</li>
<li>the coordinator can run under different budgets</li>
<li>and a human can look at the output and say:</br>
“yep, that process makes sense.”</li></ul>

That’s the recruiter-friendly “aha.”

### One Tiny Example Payload
Mini Example (One candidate plan + score)

Candidate plan (supply_plan_v1):
```json
{
  "type": "supply.plan:v1",
  "plan_id": "cand-04",
  "supplier": "VN_raw_01",
  "assembly_country": "MX",
  "ship_to": "US",
  "route": ["VN->MX", "MX->US"]
}
```
Score (supply_price_v1)
```json
{
  "type": "supply.price:v1",
  "total_score": 128.40,
  "breakdown": {
    "type": "supply.price_breakdown:v1",
    "material_cost": 50.0,
    "manufacturing_cost": 25.0,
    "shipping_cost": 30.0,
    "tariff_cost": 20.0,
    "handling_cost": 3.4,
    "total_landed_cost": 128.4
  },
  "penalties": []
}
```

### Mini “Call Graph” Summary
User → Coordinator
<ul><li>a2a.request:v1 with message_type="supply.solve:v1" and payload supply_problem_v1</li></ul>

Coordinator → Specialists
<ul>
<li>route.enumerate:v1 → returns candidate supply_plan_v1</li>
<li>plan.validate:v1 → returns pass/fail + violations</li>
<li>plan.score:v1 → returns supply_price_v1 (+ supply_price_breakdown_v1)</li></ul>

Coordinator → User
<ul><li>supply.solve.result:v1 with best plan + explanation + trace</li></ul>

Make every payload schema have:
<ul><li>type with a const (you’re already doing this)</li>
<li>$id set to the filename (you learned why the hard way 😄)</li></ul>

That gives you:
<ul><li>predictable filenames</li>
<li>resolvable $refs</li>
<li>predictable validation error paths</li>
</ul>

### Master Contract Mapping

| Schema file                                        | message_type                                          | Producer                 | Consumer                       | Why it exists                                                                                                 |
| -------------------------------------------------- | ----------------------------------------------------- | ------------------------ | ------------------------------ | ------------------------------------------------------------------------------------------------------------- |
| **a2a_request_v1.json**                            | `a2a.request:v1`                                      | Any caller               | Any callee                     | Standard “envelope” for *all* A2A requests. Gives routing + traceability.                                     |
| **a2a_response_v1.json**                           | `a2a.response:v1`                                     | Any callee               | Any caller                     | Standard “envelope” for responses. Clean ok/error semantics + consistent payload validation.                  |
| **a2a_endpoint_v1.json** *(optional)*              | (embedded object)                                     | N/A                      | N/A                            | Normalizes `source` / `dest` identity objects (name/version/url/skill). Helps debugging and “who called who.” |
| **supply_problem_v1.json**                         | `supply.solve:v1` *(or `supply.problem:v1` if split)* | User / notebook          | Coordinator                    | Defines the scenario + constraints. Prevents hand-wavy assumptions and makes runs reproducible.               |
| **supply_plan_v1.json**                            | `supply.plan:v1`                                      | Coordinator / Enumerator | Scorer, Validator, Coordinator | “State” object in search. Everything revolves around proposing/scoring/validating these.                      |
| **supply_price_breakdown_v1.json**                 | `supply.price_breakdown:v1`                           | Scorer                   | Coordinator / Report           | Line-item cost explanation. Makes scoring legible and audit-friendly.                                         |
| **supply_price_v1.json**                           | `supply.price:v1`                                     | Scorer                   | Coordinator                    | Ranking signal used by strategies. Scalar score for pruning + breakdown for humans.                           |
| **supply_trace_event_v1.json** *(recommended)*     | `supply.trace_event:v1`                               | Coordinator              | Final report                   | Standard “step log” contract: what was tried, what it cost, what got selected, and why.                       |
| **supply_strategy_config_v1.json** *(recommended)* | `supply.strategy_config:v1`                           | User / Policy            | Coordinator                    | Makes strategy choice explicit: fixed/policy/adaptive + requested vs selected strategy + budget knobs.        |
| **route_enumerate_v1.json** *(if you add it)*      | `route.enumerate:v1`                                  | Coordinator              | Enumerator agent               | Generates candidate plans. Keeps “search branching” decoupled from scoring logic.                             |
| **plan_validate_v1.json** *(if you add it)*        | `plan.validate:v1`                                    | Coordinator              | Validator agent                | Validity check is explicit, testable, and separate from scoring heuristics.                                   |
| **plan_score_v1.json** *(if you add it)*           | `plan.score:v1`                                       | Coordinator              | Scorer agent                   | Scoring request stays clean: input plan + problem context → output score + breakdown.                         |
| **supply_solve_result_v1.json** *(recommended)*    | `supply.solve.result:v1`                              | Coordinator              | User / notebook                | Final deliverable: best plan + score + trace summary. This is the “artifact” you screenshot for LinkedIn.     |
